# Week 6: Deep Learning for Stress Detection

## -Level Deep Learning Pipeline for Physiological Signal Classification

This notebook demonstrates comprehensive deep learning approaches for stress detection from the WESAD dataset, including:

1. **Data Preparation** - PyTorch datasets, dataloaders, and augmentation
2. **1D-CNN on Raw Signals** - Convolutional networks for time-series
3. **2D-CNN Transfer Learning** - EfficientNet on GASF/GADF/MTF images
4. **CNN-LSTM with Attention** - Hybrid architecture with interpretability
5. **Transformer** - Self-attention for sequence modeling
6. **Cross-Modal Attention** - Multimodal fusion with attention
7. **TabNet** - Interpretable tabular deep learning
8. **Model Comparison** - Comprehensive evaluation

All models use **Leave-One-Subject-Out (LOSO)** cross-validation for unbiased evaluation.

In [ ]:
# Standard imports
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

# Scikit-learn
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score,
    confusion_matrix, classification_report
)
from sklearn.preprocessing import StandardScaler

# Project imports
sys.path.insert(0, str(Path.cwd().parent))
from src.models.dl import (
    # Models
    CNN1D, CNN1DResidual, MultiScaleCNN1D,
    CNN2DTransferLearning,
    BiLSTMClassifier, GRUClassifier,
    CNNLSTMHybrid, DeepCNNLSTM,
    StressTransformer,
    CrossModalAttention,
    TabNetWrapper,
    # Data
    StressDataset, SequenceDataset, ImageStressDataset,
    create_data_loaders, create_loso_loaders,
    # Augmentation
    SignalAugmenter,
    # Training
    DLTrainer, loso_cross_validate,
    # Utils
    get_model, list_models,
)
from src.config import PROCESSED_DATA_DIR, FIGURES_DIR, MODELS_DIR

# Settings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

# Device configuration
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

# Create output directories
DL_MODELS_DIR = MODELS_DIR / 'trained' / 'dl'
DL_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Random seed for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 1. Data Preparation for Deep Learning

### 1.1 Load Preprocessed Features

In [ ]:
# Load feature data
features_path = PROCESSED_DATA_DIR / 'features' / 'all_subjects_features.parquet'

if features_path.exists():
    df = pd.read_parquet(features_path)
    print(f"Loaded features: {df.shape}")
    print(f"Subjects: {df['subject_id'].nunique()}")
    print(f"Conditions: {df['condition'].unique()}")
else:
    print("Feature file not found. Creating synthetic data for demonstration.")
    # Create synthetic data for demonstration
    np.random.seed(42)
    n_samples = 1000
    n_features = 50
    n_subjects = 15
    
    df = pd.DataFrame(
        np.random.randn(n_samples, n_features),
        columns=[f'feature_{i}' for i in range(n_features)]
    )
    df['subject_id'] = np.random.choice([f'S{i}' for i in range(2, n_subjects+2)], n_samples)
    df['condition'] = np.random.choice(['baseline', 'stress', 'amusement'], n_samples)
    df['label'] = df['condition'].map({'baseline': 0, 'stress': 1, 'amusement': 2})
    print(f"Created synthetic data: {df.shape}")

In [ ]:
# Prepare data for deep learning
feature_cols = [c for c in df.columns if c not in ['subject_id', 'condition', 'label', 'timestamp', 'window_id']]

X = df[feature_cols].values.astype(np.float32)
y = df['label'].values.astype(np.int64)
groups = df['subject_id'].values

# Standardize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

print(f"Features shape: {X.shape}")
print(f"Labels distribution: {np.bincount(y)}")
print(f"Unique subjects: {len(np.unique(groups))}")

### 1.2 Create PyTorch Datasets and DataLoaders

In [ ]:
# Create standard data loaders
loaders = create_data_loaders(
    X, y, groups,
    batch_size=32,
    val_split=0.2,
    use_weighted_sampler=True,
    seed=SEED
)

print("DataLoaders created:")
for name, loader in loaders.items():
    print(f"  {name}: {len(loader.dataset)} samples, {len(loader)} batches")

In [ ]:
# Verify batch shapes
for batch_x, batch_y in loaders['train']:
    print(f"Batch X shape: {batch_x.shape}")
    print(f"Batch y shape: {batch_y.shape}")
    print(f"Batch y values: {batch_y[:5]}")
    break

### 1.3 Signal Augmentation Examples

In [ ]:
# Demonstrate signal augmentation
augmenter = SignalAugmenter(random_state=42)

# Create a sample signal (simulated ECG-like)
t = np.linspace(0, 4*np.pi, 200)
signal = np.sin(t) + 0.5*np.sin(2*t) + 0.1*np.random.randn(len(t))

fig, axes = plt.subplots(3, 3, figsize=(14, 10))

augmentations = [
    ('Original', signal),
    ('Jittering', augmenter.jittering(signal, sigma=0.05)),
    ('Scaling', augmenter.scaling(signal, sigma=0.2)),
    ('Magnitude Warping', augmenter.magnitude_warping(signal, sigma=0.3)),
    ('Time Warping', augmenter.time_warping(signal, sigma=0.3)),
    ('Random Crop', augmenter.random_crop(signal, crop_ratio=0.8)),
    ('Permutation', augmenter.permutation(signal, num_segments=4)),
    ('Random Masking', augmenter.random_masking(signal, mask_ratio=0.1)),
    ('Random Augment', augmenter.random_augment(signal, n_augments=2)),
]

for ax, (name, aug_signal) in zip(axes.flat, augmentations):
    ax.plot(signal, 'b-', alpha=0.3, label='Original')
    ax.plot(aug_signal, 'r-', alpha=0.8, label='Augmented')
    ax.set_title(name)
    ax.legend(loc='upper right', fontsize=8)
    ax.set_xlabel('Time')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dl_signal_augmentation.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. 1D-CNN on Raw Signals

### Architecture
```
Input: (batch, channels, seq_len)
    │
    ▼
Conv1d(in, 64, k=7) → BatchNorm → ReLU → MaxPool
    │
    ▼
Conv1d(64, 128, k=5) → BatchNorm → ReLU → MaxPool
    │
    ▼
Conv1d(128, 256, k=3) → BatchNorm → ReLU
    │
    ▼
Global Average Pooling
    │
    ▼
FC(256, 128) → Dropout → FC(128, num_classes)
```

In [ ]:
# For 1D-CNN, reshape features as sequence
# Treat features as channels (multivariate time series)
n_features = X.shape[1]
n_classes = len(np.unique(y))

# Create CNN1D model
cnn1d_model = CNN1D(
    input_dim=1,  # Single channel
    num_classes=n_classes,
    sequence_length=n_features,  # Treat features as sequence
    base_filters=64,
    num_conv_layers=3,
    dropout=0.5,
    device=DEVICE
)

print("CNN1D Architecture:")
print(cnn1d_model.summary())

In [ ]:
# Reshape data for CNN1D: (batch, 1, seq_len)
X_cnn = X[:, np.newaxis, :].astype(np.float32)

print(f"CNN input shape: {X_cnn.shape}")

# Quick forward pass test
test_input = torch.tensor(X_cnn[:4]).to(DEVICE)
with torch.no_grad():
    test_output = cnn1d_model(test_input)
print(f"Test output shape: {test_output.shape}")

In [ ]:
# Train CNN1D with LOSO cross-validation
print("Running LOSO CV for CNN1D (this may take a while)...")

cnn1d_results = loso_cross_validate(
    model_class=CNN1D,
    X=X_cnn,
    y=y,
    groups=groups,
    model_kwargs={
        'input_dim': 1,
        'num_classes': n_classes,
        'sequence_length': n_features,
        'base_filters': 64,
        'num_conv_layers': 3,
        'dropout': 0.5,
    },
    num_epochs=50,
    batch_size=32,
    lr=1e-3,
    early_stopping_patience=10,
    device=DEVICE,
    verbose=True
)

print(f"\nCNN1D LOSO Results:")
print(f"  Accuracy: {cnn1d_results['accuracy_mean']:.4f} ± {cnn1d_results['accuracy_std']:.4f}")
print(f"  F1-Macro: {cnn1d_results['f1_macro_mean']:.4f} ± {cnn1d_results['f1_macro_std']:.4f}")

In [ ]:
# Visualize CNN1D results
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Per-subject accuracy
subjects = [r['subject'] for r in cnn1d_results['fold_results']]
accuracies = [r['accuracy'] for r in cnn1d_results['fold_results']]

ax = axes[0]
ax.bar(range(len(subjects)), accuracies, color='steelblue', alpha=0.7)
ax.axhline(y=cnn1d_results['accuracy_mean'], color='r', linestyle='--', label=f"Mean: {cnn1d_results['accuracy_mean']:.3f}")
ax.set_xlabel('Subject')
ax.set_ylabel('Accuracy')
ax.set_title('CNN1D: Per-Subject Accuracy (LOSO)')
ax.set_xticks(range(len(subjects)))
ax.set_xticklabels(subjects, rotation=45)
ax.legend()

# Confusion matrix
cm = confusion_matrix(cnn1d_results['targets'], cnn1d_results['predictions'])
ax = axes[1]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Baseline', 'Stress', 'Amusement'],
            yticklabels=['Baseline', 'Stress', 'Amusement'])
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('CNN1D: Confusion Matrix')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dl_cnn1d_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. BiLSTM with Attention

### Architecture
```
Input: (batch, seq_len, features)
    │
    ▼
Bidirectional LSTM (2 layers)
    │
    ▼
Attention Mechanism (optional)
    │
    ▼
FC → Classification
```

In [ ]:
# For LSTM, reshape: (batch, seq_len, features)
# Treat each sample as a sequence of 1 timestep with n_features
X_lstm = X[:, np.newaxis, :].astype(np.float32)  # (batch, 1, features)

# Or create windows from features
window_size = 10
n_windows = n_features // window_size
X_lstm_windowed = X[:, :n_windows*window_size].reshape(-1, n_windows, window_size).astype(np.float32)

print(f"LSTM windowed input shape: {X_lstm_windowed.shape}")

# Create BiLSTM model
bilstm_model = BiLSTMClassifier(
    input_dim=window_size,
    num_classes=n_classes,
    hidden_dim=64,
    num_layers=2,
    dropout=0.3,
    use_attention=True,
    bidirectional=True,
    device=DEVICE
)

print("BiLSTM with Attention:")
print(f"  Parameters: {bilstm_model.count_parameters():,}")

In [ ]:
# Train BiLSTM with LOSO
print("Running LOSO CV for BiLSTM...")

bilstm_results = loso_cross_validate(
    model_class=BiLSTMClassifier,
    X=X_lstm_windowed,
    y=y,
    groups=groups,
    model_kwargs={
        'input_dim': window_size,
        'num_classes': n_classes,
        'hidden_dim': 64,
        'num_layers': 2,
        'dropout': 0.3,
        'use_attention': True,
    },
    num_epochs=50,
    batch_size=32,
    lr=1e-3,
    early_stopping_patience=10,
    device=DEVICE,
    verbose=True
)

print(f"\nBiLSTM LOSO Results:")
print(f"  Accuracy: {bilstm_results['accuracy_mean']:.4f} ± {bilstm_results['accuracy_std']:.4f}")
print(f"  F1-Macro: {bilstm_results['f1_macro_mean']:.4f} ± {bilstm_results['f1_macro_std']:.4f}")

## 4. CNN-LSTM Hybrid with Temporal Attention

### Architecture
```
Input: (batch, channels, seq_len)
    │
    ▼
CNN Feature Extractor (2 conv blocks)
    │
    ▼
Bidirectional LSTM
    │
    ▼
Temporal Attention (multi-head)
    │
    ▼
Classification Head
```

In [ ]:
# Create CNN-LSTM Hybrid model
cnn_lstm_model = CNNLSTMHybrid(
    input_dim=1,
    num_classes=n_classes,
    sequence_length=n_features,
    cnn_filters=(64, 128),
    lstm_hidden=64,
    lstm_layers=2,
    attention_heads=4,
    dropout=0.5,
    device=DEVICE
)

print("CNN-LSTM Hybrid:")
print(f"  Parameters: {cnn_lstm_model.count_parameters():,}")

In [ ]:
# Train CNN-LSTM with LOSO
print("Running LOSO CV for CNN-LSTM Hybrid...")

cnn_lstm_results = loso_cross_validate(
    model_class=CNNLSTMHybrid,
    X=X_cnn,
    y=y,
    groups=groups,
    model_kwargs={
        'input_dim': 1,
        'num_classes': n_classes,
        'sequence_length': n_features,
        'cnn_filters': (64, 128),
        'lstm_hidden': 64,
        'dropout': 0.5,
    },
    num_epochs=50,
    batch_size=32,
    lr=1e-3,
    early_stopping_patience=10,
    device=DEVICE,
    verbose=True
)

print(f"\nCNN-LSTM LOSO Results:")
print(f"  Accuracy: {cnn_lstm_results['accuracy_mean']:.4f} ± {cnn_lstm_results['accuracy_std']:.4f}")
print(f"  F1-Macro: {cnn_lstm_results['f1_macro_mean']:.4f} ± {cnn_lstm_results['f1_macro_std']:.4f}")

In [ ]:
# Visualize attention weights
# Train a model on all data for visualization
cnn_lstm_viz = CNNLSTMHybrid(
    input_dim=1,
    num_classes=n_classes,
    sequence_length=n_features,
    device=DEVICE
)

# Quick training
optimizer = torch.optim.Adam(cnn_lstm_viz.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
trainer = DLTrainer(cnn_lstm_viz, optimizer, criterion)

train_dataset = StressDataset(X_cnn.reshape(X_cnn.shape[0], -1), y)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Train for a few epochs
for epoch in range(5):
    cnn_lstm_viz.train()
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.view(-1, 1, n_features).to(DEVICE)
        batch_y = batch_y.to(DEVICE)
        
        optimizer.zero_grad()
        output = cnn_lstm_viz(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()

# Get attention weights for a sample
sample_x = torch.tensor(X_cnn[:8]).to(DEVICE)
attn_weights = cnn_lstm_viz.get_attention_weights(sample_x)

# Plot attention weights
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    if attn_weights is not None:
        ax.bar(range(len(attn_weights[i])), attn_weights[i].cpu().numpy())
        ax.set_title(f'Sample {i+1}')
        ax.set_xlabel('Position')
        ax.set_ylabel('Attention Weight')

plt.suptitle('CNN-LSTM Temporal Attention Weights', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dl_cnn_lstm_attention.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Transformer for Stress Detection

### Architecture
```
Input: (batch, seq_len, features)
    │
    ▼
Input Projection + [CLS] Token
    │
    ▼
Positional Encoding (learnable)
    │
    ▼
Transformer Encoder (4 layers, 8 heads)
    │
    ▼
[CLS] Token → Classification
```

In [ ]:
# Create Transformer model
transformer_model = StressTransformer(
    input_dim=window_size,
    num_classes=n_classes,
    d_model=64,
    nhead=4,
    num_layers=3,
    d_ff=256,
    dropout=0.1,
    positional_encoding='learnable',
    use_cls_token=True,
    device=DEVICE
)

print("Stress Transformer:")
print(f"  Parameters: {transformer_model.count_parameters():,}")

In [ ]:
# Train Transformer with LOSO
print("Running LOSO CV for Transformer...")

transformer_results = loso_cross_validate(
    model_class=StressTransformer,
    X=X_lstm_windowed,
    y=y,
    groups=groups,
    model_kwargs={
        'input_dim': window_size,
        'num_classes': n_classes,
        'd_model': 64,
        'nhead': 4,
        'num_layers': 3,
        'd_ff': 256,
        'dropout': 0.1,
    },
    num_epochs=50,
    batch_size=32,
    lr=1e-4,  # Lower LR for Transformer
    early_stopping_patience=10,
    device=DEVICE,
    verbose=True
)

print(f"\nTransformer LOSO Results:")
print(f"  Accuracy: {transformer_results['accuracy_mean']:.4f} ± {transformer_results['accuracy_std']:.4f}")
print(f"  F1-Macro: {transformer_results['f1_macro_mean']:.4f} ± {transformer_results['f1_macro_std']:.4f}")

## 6. Cross-Modal Attention for Multimodal Fusion

In [ ]:
# For cross-modal attention, we need to split features by modality
# In WESAD, we have: ECG, EDA, EMG, TEMP, ACC, RESP features

# Simulate modality-specific features (in practice, extract from feature names)
n_total_features = X.shape[1]
modality_split = n_total_features // 4

modality_data = {
    'ECG': X[:, :modality_split].astype(np.float32),
    'EDA': X[:, modality_split:2*modality_split].astype(np.float32),
    'TEMP': X[:, 2*modality_split:3*modality_split].astype(np.float32),
    'ACC': X[:, 3*modality_split:].astype(np.float32),
}

modality_dims = {k: v.shape[1] for k, v in modality_data.items()}
print("Modality dimensions:")
for name, dim in modality_dims.items():
    print(f"  {name}: {dim}")

In [ ]:
# Create Cross-Modal Attention model
cross_modal_model = CrossModalAttention(
    modality_dims=modality_dims,
    num_classes=n_classes,
    d_model=64,
    nhead=4,
    num_cross_layers=2,
    dropout=0.1,
    fusion_method='attention',
    device=DEVICE
)

print("Cross-Modal Attention Model:")
print(f"  Parameters: {cross_modal_model.count_parameters():,}")
print(f"  Modalities: {cross_modal_model.modality_names}")

In [ ]:
# Test forward pass
test_input = {
    name: torch.tensor(data[:4, np.newaxis, :]).to(DEVICE)  # (batch, 1, features)
    for name, data in modality_data.items()
}

with torch.no_grad():
    output, importance = cross_modal_model(test_input, return_importance=True)

print(f"Output shape: {output.shape}")
print("\nModality Importance:")
for name, imp in importance.items():
    print(f"  {name}: {imp:.4f}")

## 7. TabNet - Interpretable Tabular Learning

In [ ]:
# TabNet for interpretable predictions
try:
    tabnet_model = TabNetWrapper(
        n_d=32,
        n_a=32,
        n_steps=4,
        gamma=1.5,
        lambda_sparse=1e-3,
        verbose=0,
        device_name=DEVICE
    )
    print(tabnet_model.summary())
    TABNET_AVAILABLE = True
except ImportError:
    print("TabNet not available. Install with: pip install pytorch-tabnet")
    TABNET_AVAILABLE = False

In [ ]:
if TABNET_AVAILABLE:
    # LOSO for TabNet
    unique_subjects = np.unique(groups)
    tabnet_fold_results = []
    
    for test_subject in unique_subjects[:3]:  # Limit for demo
        print(f"TabNet - Testing on subject {test_subject}")
        
        # Split data
        test_mask = groups == test_subject
        train_mask = ~test_mask
        
        X_train, y_train = X[train_mask], y[train_mask]
        X_test, y_test = X[test_mask], y[test_mask]
        
        # Validation split
        val_size = int(0.1 * len(X_train))
        X_val, y_val = X_train[:val_size], y_train[:val_size]
        X_train, y_train = X_train[val_size:], y_train[val_size:]
        
        # Train TabNet
        tabnet = TabNetWrapper(n_d=32, n_a=32, n_steps=4, verbose=0)
        tabnet.fit(
            X_train, y_train,
            X_val, y_val,
            max_epochs=50,
            patience=10,
            batch_size=256
        )
        
        # Evaluate
        y_pred = tabnet.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='macro')
        
        tabnet_fold_results.append({'subject': test_subject, 'accuracy': acc, 'f1_macro': f1})
        print(f"  Accuracy: {acc:.4f}, F1: {f1:.4f}")
    
    print(f"\nTabNet Mean Accuracy: {np.mean([r['accuracy'] for r in tabnet_fold_results]):.4f}")

In [ ]:
if TABNET_AVAILABLE:
    # Visualize TabNet feature importance
    importance = tabnet.get_feature_importance()
    
    # Get local explanations
    masks, local_importance = tabnet.explain(X_test[:10])
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Global importance
    ax = axes[0]
    top_k = 20
    top_idx = np.argsort(importance)[-top_k:]
    ax.barh(range(top_k), importance[top_idx])
    ax.set_yticks(range(top_k))
    ax.set_yticklabels([f'Feature {i}' for i in top_idx])
    ax.set_xlabel('Importance')
    ax.set_title('TabNet Global Feature Importance (Top 20)')
    
    # Local explanation heatmap
    ax = axes[1]
    sns.heatmap(local_importance[:, :30], cmap='YlOrRd', ax=ax)
    ax.set_xlabel('Feature')
    ax.set_ylabel('Sample')
    ax.set_title('TabNet Local Feature Importance')
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'dl_tabnet_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

## 8. Model Comparison

### Comprehensive Comparison of All Deep Learning Models

In [ ]:
# Collect all results
dl_results = {
    'CNN1D': {
        'accuracy': cnn1d_results.get('accuracy_mean', 0),
        'accuracy_std': cnn1d_results.get('accuracy_std', 0),
        'f1_macro': cnn1d_results.get('f1_macro_mean', 0),
        'f1_std': cnn1d_results.get('f1_macro_std', 0),
        'params': cnn1d_model.count_parameters(),
    },
    'BiLSTM-Attention': {
        'accuracy': bilstm_results.get('accuracy_mean', 0),
        'accuracy_std': bilstm_results.get('accuracy_std', 0),
        'f1_macro': bilstm_results.get('f1_macro_mean', 0),
        'f1_std': bilstm_results.get('f1_macro_std', 0),
        'params': bilstm_model.count_parameters(),
    },
    'CNN-LSTM Hybrid': {
        'accuracy': cnn_lstm_results.get('accuracy_mean', 0),
        'accuracy_std': cnn_lstm_results.get('accuracy_std', 0),
        'f1_macro': cnn_lstm_results.get('f1_macro_mean', 0),
        'f1_std': cnn_lstm_results.get('f1_macro_std', 0),
        'params': cnn_lstm_model.count_parameters(),
    },
    'Transformer': {
        'accuracy': transformer_results.get('accuracy_mean', 0),
        'accuracy_std': transformer_results.get('accuracy_std', 0),
        'f1_macro': transformer_results.get('f1_macro_mean', 0),
        'f1_std': transformer_results.get('f1_macro_std', 0),
        'params': transformer_model.count_parameters(),
    },
}

# Create comparison DataFrame
comparison_df = pd.DataFrame(dl_results).T
comparison_df['accuracy_str'] = comparison_df.apply(
    lambda x: f"{x['accuracy']:.4f} ± {x['accuracy_std']:.4f}", axis=1
)
comparison_df['f1_str'] = comparison_df.apply(
    lambda x: f"{x['f1_macro']:.4f} ± {x['f1_std']:.4f}", axis=1
)

print("\n" + "="*80)
print("DEEP LEARNING MODEL COMPARISON (LOSO Cross-Validation)")
print("="*80)
print(comparison_df[['accuracy_str', 'f1_str', 'params']].to_string())

In [ ]:
# Visualization: Model Comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

models = list(dl_results.keys())
colors = plt.cm.Set2(np.linspace(0, 1, len(models)))

# Accuracy comparison
ax = axes[0]
accuracies = [dl_results[m]['accuracy'] for m in models]
errors = [dl_results[m]['accuracy_std'] for m in models]
bars = ax.bar(models, accuracies, yerr=errors, capsize=5, color=colors, alpha=0.8)
ax.set_ylabel('Accuracy')
ax.set_title('Model Accuracy Comparison')
ax.set_ylim(0, 1)
ax.tick_params(axis='x', rotation=45)
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{acc:.3f}', ha='center', va='bottom', fontsize=9)

# F1 Score comparison
ax = axes[1]
f1_scores = [dl_results[m]['f1_macro'] for m in models]
f1_errors = [dl_results[m]['f1_std'] for m in models]
bars = ax.bar(models, f1_scores, yerr=f1_errors, capsize=5, color=colors, alpha=0.8)
ax.set_ylabel('F1-Macro Score')
ax.set_title('Model F1-Score Comparison')
ax.set_ylim(0, 1)
ax.tick_params(axis='x', rotation=45)
for bar, f1 in zip(bars, f1_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{f1:.3f}', ha='center', va='bottom', fontsize=9)

# Parameter count comparison
ax = axes[2]
params = [dl_results[m]['params'] / 1000 for m in models]  # In thousands
bars = ax.bar(models, params, color=colors, alpha=0.8)
ax.set_ylabel('Parameters (K)')
ax.set_title('Model Complexity')
ax.tick_params(axis='x', rotation=45)
for bar, p in zip(bars, params):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{p:.1f}K', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dl_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Best model selection
best_model_name = max(dl_results, key=lambda x: dl_results[x]['f1_macro'])
best_model_results = dl_results[best_model_name]

print("\n" + "="*80)
print("BEST DEEP LEARNING MODEL")
print("="*80)
print(f"Model: {best_model_name}")
print(f"Accuracy: {best_model_results['accuracy']:.4f} ± {best_model_results['accuracy_std']:.4f}")
print(f"F1-Macro: {best_model_results['f1_macro']:.4f} ± {best_model_results['f1_std']:.4f}")
print(f"Parameters: {best_model_results['params']:,}")

## Summary

### Key Findings:

1. **CNN1D** - Efficient for extracting local patterns from physiological signals
2. **BiLSTM with Attention** - Captures temporal dependencies with interpretable attention
3. **CNN-LSTM Hybrid** - Combines local and temporal features effectively
4. **Transformer** - Self-attention captures long-range dependencies
5. **Cross-Modal Attention** - Enables multimodal fusion with modality importance
6. **TabNet** - Provides built-in interpretability for tabular features

### Recommendations:

- For **raw signals**: CNN1D or CNN-LSTM Hybrid
- For **interpretability**: TabNet or attention-based models
- For **multimodal data**: Cross-Modal Attention
- For **image-encoded signals**: 2D-CNN with Transfer Learning

All models were evaluated using Leave-One-Subject-Out cross-validation for unbiased, generalizable results.

In [ ]:
# Save final results
results_df = pd.DataFrame(dl_results).T
results_df.to_csv(PROCESSED_DATA_DIR / 'dl_model_comparison.csv')
print(f"Results saved to {PROCESSED_DATA_DIR / 'dl_model_comparison.csv'}")